In [1]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://afisha.yandex.ru/samara/events?page=40")

input() # как страница прогрузится нажать

source = driver.page_source

driver.quit()

In [78]:
from bs4 import BeautifulSoup as BS
import json
soup = BS(source, features = "lxml")
for s in soup.find_all( class_= "events-list__item" ):
    print(s)
    j = json.loads(s.select_one(".event-card-react")["data-bem"])
    break

<div class="event events-list__item yandex-sans"><div class="i-react event-card-react i-bem event-card-react_js_inited" data-bem='{"event-card-react":{"props":{"size":"l","link":"/samara/concert/macan-tour?source=rubric","id":"65ca118c7c8247006def3b78","type":"concert","tag":"quest-players-8","hasFreeYaPlusTag":false,"isEventTicketsAreFree":false,"image":{"url":"https://avatars.mds.yandex.net/get-afishanew/21626/73f3b3906e665d51e155c14bf13a52e8/s380x220","baseColor":"#c4c4c4","retina":{"1x":"https://avatars.mds.yandex.net/get-afishanew/21626/73f3b3906e665d51e155c14bf13a52e8/s380x220","2x":"https://avatars.mds.yandex.net/get-afishanew/21626/73f3b3906e665d51e155c14bf13a52e8/s760x440"}},"title":"Macan","argument":"I Am Tour","ageLimit":"16+","additionalInfo":"25 мая, 20:00","place":{"title":"СКК Дворец спорта","id":"575aad267a10ff694f6d12fa","url":"/samara/sport/places/skk-dvorets-sporta","type":{"id":"5575d419cc1c724f5a8e1150","rubricUrl":"/samara/stadium","rubricPlacesUrl":"/samara/stad

In [3]:
from bs4 import BeautifulSoup as BS
import json
import dateparser

events = []
locations = {}

def location(lon, lat):
    return "latitude=%f;longitude=%f" % (lat, lon)

soup = BS(source, features = "lxml")
for s in soup.find_all( class_= "events-list__item" ):
    j = json.loads(s.select_one(".event-card-react")["data-bem"])
    props = j["event-card-react"]["props"]
    url = "https://afisha.yandex.ru" + props["link"]
    
    event = {
        "title": props["title"],
        "source_url": url,
        "description": props["title"],
        "price": props.get("ticketsPrice"),
        "cover_img_url": None,
        "location_id": None,
        "date": None
    }
    #print(props["type"])
    #print(props["tag"])
    if props.get("image"):
        if(props["image"].get("retina")):
            event["cover_img_url"] = props["image"]["retina"][list(props["image"]["retina"].keys())[-1]]
        else:
            event["cover_img_url"] = props["image"]["url"]

    #print(props["ageLimit"])
    if props.get("place") and props["place"].get("coordinates"):
        place = props["place"]
        place_id = int(place["id"], 16) % (1<<63)
        event["location_id"] = place_id
        
        if not locations.get(place_id):
            loc = {
                "id": place_id,
                "title": place["title"],
                "address": place["address"],
            }

            if place.get("coordinates"):
                loc.update(place["coordinates"])
            else:
                loc.update({"latitude": None, "longitude": None})
            #todo: city

            locations[place_id] = loc
    else:
        event["location_id"] = None
    
    if props.get("additionalInfo"):
        date = dateparser.parse(props["additionalInfo"])
        if date:
            event["date"] = str(date)

    events.append(event)



C:\Users\KOT\AppData\Local\Temp\ipykernel_11268\2835781005.py:58: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  date = dateparser.parse(props["additionalInfo"])


In [4]:
from selenium import webdriver
import json
from time import sleep

driver = webdriver.Chrome()

events2 = []

for event in events:
    if(any(map(lambda ev: event["source_url"] == ev["source_url"], events2))): continue
    driver.get(event["source_url"])
    source = driver.page_source

    soup = BS(source, features = "lxml")

    event["_tags"] = set(map(lambda li: li.text, soup.select(".tags > li")))

    data = json.loads(soup.find(type="application/ld+json").text)[0]
    event["description"] = data.get("description")
    #event["cover_img_url"] = data.get("image")

    events2.append(event)

driver.quit()

In [5]:
def wrap(item):
    if isinstance(item, str):
        return "'%s'" % item.replace("'", "''")
    if item is None:
        return "NULL"
    return str(item)

def wrapper(container):
    return "(%s)" % ",\n".join(list(map(wrap, container)))

def dbPopulator(filename, table_name, data):
    f = open(filename, "w", encoding="utf-8")

    kys = tuple(filter(lambda k: not k.startswith("_"), next(iter(data)).keys()))

    f.write(f"INSERT INTO {table_name} ")

    f.write("(%s)" % ",".join(kys))

    f.write("\nVALUES ")

    f.write(",\n".join([wrapper(list(map(lambda x: x[1], filter(lambda l: l[0] in kys, d.items())))) for d in data]))

    f.write(";\n")

    f.write("ALTER SEQUENCE IF EXISTS %s RESTART WITH %d;\n" % (table_name + "_id_seq", len(data) + 1))

    f.close()

events3 = list(map(lambda e: e[1] | {"id": e[0]}, enumerate(events2, start=1)))

tags = set()
for e in events3:
    tags |= e.get("_tags", set())

tags_data = []
tags_reverse_ids = {}

for i, v in enumerate(tags, start=1):
    tags_data.append({"id": i, "name": v})
    tags_reverse_ids[v] = i

tags_events = []
for e in events3:
    for t in e.get("_tags", set()):
        tags_events.append({"event_id": e["id"], "tag_id": tags_reverse_ids[t]})

dbPopulator("1populate_locations.sql", "locations", locations.values())
dbPopulator("2populate_tags.sql", "tags", tags_data)
dbPopulator("3populate_events.sql", "events", events3)
dbPopulator("4populate_event_tags.sql", "event_tags", tags_events)
